In [2]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles()
styles.register_and_enable_theme()

# ---------- data ----------
IDS = {
    "IRLTLT01DEM156N": "Germany", "IRLTLT01FRM156N": "France", "IRLTLT01ITM156N": "Italy",
    "IRLTLT01ESM156N": "Spain", "IRLTLT01NLM156N": "Netherlands", "IRLTLT01BEM156N": "Belgium",
}
ORDER = ["France", "Italy", "Spain", "Netherlands", "Belgium"]

raw = pd.read_csv("../data/fredgraph.csv", parse_dates=["observation_date"]).rename(
    columns={"observation_date": "date", **IDS}
)
raw = raw.query("date >= '2010-01-01'")

# spread = country 10y yield minus German 10y yield (percentage points)
wide = raw[["date"]].copy()
for c in ORDER:
    wide[c] = raw[c] - raw["Germany"]
long = wide.melt("date", var_name="country", value_name="spread").dropna()

# ---------- hover: snap to nearest month ----------
nearest = alt.selection_point(
    fields=["date"], nearest=True, on="pointerover", clear="pointerout", empty=False
)

# ---------- encodings ----------
legend = alt.Legend(
    title=None, orient="bottom", direction="horizontal", columns=5,
    symbolType="stroke", symbolStrokeWidth=2, symbolSize=120,
    labelFontSize=10, columnPadding=16, offset=8,
)
color = alt.Color("country:N", sort=ORDER, legend=legend)
x = alt.X("date:T", title=None, axis=alt.Axis(format="%Y", tickCount=9))
y = alt.Y("spread:Q", title="pp")

# ---------- layers ----------
lines = alt.Chart(long).mark_line(strokeWidth=1.8).encode(x=x, y=y, color=color)

dots = (
    alt.Chart(long).mark_circle(size=45)
    .encode(x=x, y=y, color=color, opacity=alt.condition(nearest, alt.value(1), alt.value(0)))
)

# tooltip: every country's spread for the hovered month
tooltip = [alt.Tooltip("date:T", title="Month", format="%b %Y")] + [
    alt.Tooltip(f"{c}:Q", title=c, format=".2f") for c in ORDER
]
hover_band = (
    alt.Chart(wide).mark_rule(color="#94a3b8", strokeWidth=8, opacity=0.25)
    .encode(x="date:T", tooltip=tooltip)
    .transform_filter(nearest)
)
catcher = alt.Chart(wide).mark_point(opacity=0).encode(x="date:T").add_params(nearest)

chart = (
    (catcher + hover_band + lines + dots)
    .properties(
        width=400, height=300,
        title=alt.Title(
            "Spreads of 10-year government bond yields over Germany, 2010–26",
            subtitle="Monthly 10-year yield minus German 10-year yield, percentage points. Source: FRED",
        ),
    )
)

styles.save(chart, path=".", name="bond_spreads", svg=True)
chart

alt.LayerChart(...)

In [4]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles()
styles.register_and_enable_theme()

# ---------- data ----------
IDS = {
    "IRLTLT01DEM156N": "Germany", "IRLTLT01FRM156N": "France", "IRLTLT01ITM156N": "Italy",
    "IRLTLT01ESM156N": "Spain", "IRLTLT01NLM156N": "Netherlands", "IRLTLT01BEM156N": "Belgium",
}
ORDER = ["France", "Italy", "Spain", "Netherlands", "Belgium"]

raw = pd.read_csv("../data/fredgraph.csv", parse_dates=["observation_date"]).rename(
    columns={"observation_date": "date", **IDS}
)
raw = raw.query("date >= '2010-01-01'")

# spread = country 10y yield minus German 10y yield (percentage points)
wide = raw[["date"]].copy()
for c in ORDER:
    wide[c] = raw[c] - raw["Germany"]
long = wide.melt("date", var_name="country", value_name="spread").dropna()

# ---------- hover: snap to nearest month ----------
nearest = alt.selection_point(
    fields=["date"], nearest=True, on="pointerover", clear="pointerout", empty=False
)

# ---------- encodings ----------
legend = alt.Legend(
    title=None, orient="bottom", direction="horizontal", columns=5,
    symbolType="stroke", symbolStrokeWidth=2, symbolSize=120,
    labelFontSize=10, columnPadding=16, offset=8,
)
color = alt.Color("country:N", sort=ORDER, legend=legend)
x = alt.X("date:T", title=None, axis=alt.Axis(format="%Y", tickCount=9))
y = alt.Y("spread:Q", title="pp")

# ---------- layers ----------
lines = alt.Chart(long).mark_line(strokeWidth=1.8).encode(x=x, y=y, color=color)

dots = (
    alt.Chart(long).mark_circle(size=45)
    .encode(x=x, y=y, color=color, opacity=alt.condition(nearest, alt.value(1), alt.value(0)))
)

# tooltip: every country's spread for the hovered month
tooltip = [alt.Tooltip("date:T", title="Month", format="%b %Y")] + [
    alt.Tooltip(f"{c}:Q", title=c, format=".2f") for c in ORDER
]
hover_band = (
    alt.Chart(wide).mark_rule(color="#94a3b8", strokeWidth=8, opacity=0.25)
    .encode(x="date:T", tooltip=tooltip)
    .transform_filter(nearest)
)
catcher = alt.Chart(wide).mark_point(opacity=0).encode(x="date:T").add_params(nearest)

chart = (
    (catcher + hover_band + lines + dots)
    .properties(
        width=400, height=300,

    )
)

styles.save(chart, path=".", name="bond_spreads", svg=True)
chart

alt.LayerChart(...)

In [5]:
raw = pd.read_csv("../data/fredgraph.csv", parse_dates=["observation_date"])
print(raw["observation_date"].max())

cols = ["IRLTLT01DEM156N", "IRLTLT01FRM156N", "IRLTLT01ITM156N",
        "IRLTLT01ESM156N", "IRLTLT01NLM156N", "IRLTLT01BEM156N"]
tail = raw.set_index("observation_date")[cols].tail(4)
spreads = tail.drop(columns="IRLTLT01DEM156N").sub(tail["IRLTLT01DEM156N"], axis=0)
spreads.columns = ["France", "Italy", "Spain", "Netherlands", "Belgium"]
print(spreads.round(2))

2026-09-01 00:00:00
                  France  Italy  Spain  Netherlands  Belgium
observation_date                                            
2026-06-01          0.71   0.77   0.45         0.11     0.55
2026-07-01          0.81   0.81   0.46         0.12     0.57
2026-08-01          0.85   0.80   0.45         0.10     0.57
2026-09-01          1.01    NaN   0.45          NaN      NaN


In [28]:
import numpy as np
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles()
styles.register_and_enable_theme()

# ---------- FRED monthly data (history) ----------
IDS = {
    "IRLTLT01DEM156N": "Germany", "IRLTLT01FRM156N": "France", "IRLTLT01ITM156N": "Italy",
    "IRLTLT01ESM156N": "Spain", "IRLTLT01NLM156N": "Netherlands", "IRLTLT01BEM156N": "Belgium",
}
ORDER = ["France", "Italy", "Spain", "Netherlands", "Belgium"]

raw = pd.read_csv("../data/fredgraph.csv", parse_dates=["observation_date"]).rename(
    columns={"observation_date": "date", **IDS}
)
raw = raw.query("date >= '2010-01-01'")

# spread = country 10y yield minus German 10y yield (percentage points)
wide = raw[["date"]].copy()
for c in ORDER:
    wide[c] = raw[c] - raw["Germany"]

# ---------- LSEG daily data: fills Sept gaps and adds October month-to-date ----------
LSEG = {
    "Date": "date", "DE10YT=RR (BID_YIELD)": "Germany", "FR10YT=RR (BID_YIELD)": "France",
    "IT10YT=RR (BID_YIELD)": "Italy", "NL10YT=RR (BID_YIELD)": "Netherlands",
    "BE10YT=RR (BID_YIELD)": "Belgium", "ES10YT=RR (BID_YIELD)": "Spain",
}
daily = (
    pd.read_excel("../data/Paul (3).xlsx", skiprows=[1])        # row 2 is just a "Close" label row
    .rename(columns=LSEG)[list(LSEG.values())]
    .assign(date=lambda d: pd.to_datetime(d["date"]))
    .set_index("date").sort_index()
)
monthly = daily.resample("MS").mean()                         # monthly average of daily yields
lseg = pd.DataFrame({c: monthly[c] - monthly["Germany"] for c in ORDER})
lseg = lseg.loc["2026-09-01":]                                # only the months FRED is missing/incomplete

# FRED values win where they exist; LSEG fills the gaps and adds Oct 2026
wide = wide.set_index("date").combine_first(lseg).sort_index().reset_index()
long = wide.melt("date", var_name="country", value_name="spread").dropna()

# ---------- hover: snap to nearest month ----------
nearest = alt.selection_point(
    fields=["date"], nearest=True, on="pointerover", clear="pointerout", empty=False
)

# ---------- encodings ----------
legend = alt.Legend(
    title=None, orient="bottom", direction="horizontal", columns=5,
    symbolType="stroke", symbolStrokeWidth=2, symbolSize=120,
    labelFontSize=10, columnPadding=16, offset=8,
)
color = alt.Color("country:N", sort=ORDER, legend=legend)
x = alt.X("date:T", title=None, axis=alt.Axis(format="%Y", tickCount=9))
y = alt.Y("spread:Q", title="pp")

# ---------- layers ----------
lines = alt.Chart(long).mark_line(strokeWidth=1.8).encode(x=x, y=y, color=color)

dots = (
    alt.Chart(long).mark_circle(size=45)
    .encode(x=x, y=y, color=color, opacity=alt.condition(nearest, alt.value(1), alt.value(0)))
)

tooltip = [alt.Tooltip("date:T", title="Month", format="%b %Y")] + [
    alt.Tooltip(f"{c}:Q", title=c, format=".2f") for c in ORDER
]
hover_band = (
    alt.Chart(wide).mark_rule(color="#94a3b8", strokeWidth=8, opacity=0.25)
    .encode(x="date:T", tooltip=tooltip)
    .transform_filter(nearest)
)
catcher = alt.Chart(wide).mark_point(opacity=0).encode(x="date:T").add_params(nearest)

# ---------- highlight: ellipse around the October upturn ----------
last = long["date"].max()
ends = long.loc[(long["date"] == last) & (long["country"] != "Netherlands"), "spread"]
t = np.linspace(0, 2 * np.pi, 120)
ring_df = pd.DataFrame({
    "t": t,
    "date": (last - pd.Timedelta(days=100)) + pd.to_timedelta(150 * np.cos(t), unit="D"),
    "spread": (ends.min() + ends.max()) / 2 + ((ends.max() - ends.min()) / 2 + 0.25) * np.sin(t),
})
ring = (
    alt.Chart(ring_df).mark_line(color="orange", strokeWidth=1.5)
    .encode(x=x, y=y, order="t:Q")
)

chart = (
    (catcher + hover_band + lines + dots)
    .properties(
        width=400, height=300,
       
    )
)

styles.save(chart, path=".", name="bond_spreads", svg=True)
chart

alt.LayerChart(...)

In [15]:
print(long.columns.tolist())

['date', 'country', 'spread']
